In [ ]:
import gymnasium as gym
from tqdm import tqdm 
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
import os
import gymnasium as gym
import matplotlib.pyplot as plt

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.results_plotter import load_results, ts2xy
import pandas as pd 

In [ ]:
log_dir = "./racing/logs/"
results_dir = "./racing/results/"

num_timesteps = 1000

## Train through PPO

In [ ]:

os.makedirs(log_dir, exist_ok=True)

vec_env = make_vec_env(
    "CarRacing-v3",
    monitor_dir=log_dir,
    n_envs=4, 
    seed=42, 
    env_kwargs={"continuous": False}
)

model = PPO(
    "CnnPolicy",
    vec_env,
    verbose=1,
    seed=42, 
    device="mps"
)

model.learn(total_timesteps=num_timesteps)
model.save("ppo_racing")
vec_env.close()
print("finished training")

## Plotting training logs 

In [ ]:
from stable_baselines3.common.results_plotter import plot_results
from stable_baselines3.common import results_plotter

os.makedirs(results_dir, exist_ok=True)

plot_results([log_dir], None, results_plotter.X_TIMESTEPS, "PPO Car - Rewards vs. Timestep")

plot_results([log_dir], None, results_plotter.X_EPISODES, "Rewards vs. Episode")

plot_results([log_dir], None, results_plotter.X_WALLTIME, "Rewards vs. Time")


## Test trained model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

model = PPO.load("ppo_racing")

eval_env = gym.make(
    "CarRacing-v3",
    continuous=False,
    render_mode="human"
)

total_rewards = []
completion_results = []

num_episodes = 5

for episode in range(num_episodes):

    print("episode: ", episode)
    obs, info = eval_env.reset()

    episode_reward = 0
    done = False

    while not done:
        action, _ = model.predict(obs, deterministic=True)

        obs, reward, terminated, truncated, info = eval_env.step(action)

        episode_reward += reward
        done = terminated or truncated

    total_rewards.append(episode_reward)

    completion_results.append(episode_reward > 0)

eval_env.close()

print(f"Average Reward: {np.mean(total_rewards):.3f}")
print(f"Standard Deviation: {np.std(total_rewards):.3f}")
print(f"Completion Rate (reward proxy): {np.mean(completion_results):.1%}")

episodes = np.arange(1, num_episodes + 1)

plt.figure(figsize=(10, 5))

plt.plot(
    episodes,
    total_rewards,
    marker="o",
    label="Episode Reward"
)

plt.axhline(
    np.mean(total_rewards),
    linestyle="--",
    label=f"Mean Reward: {np.mean(total_rewards):.2f}"
)

plt.xlabel("Evaluation Episode")
plt.ylabel("Total Episode Reward")
plt.title("PPO CarRacing: Evaluation Rewards")
plt.xticks(episodes)
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()